# etl_pipeline.ipynb
## Drilling Operations — ETL Pipeline

Extracts data from `DataForAssessment.csv`, transforms it into the 9-table normalised
schema, and loads it into `drilling_operations.db`.

| Stage | Description |
|---|---|
| **Extract** | Read CSV, inspect shape/dtypes, flag missing values |
| **Transform** | Deduplicate, cast types, normalise datetimes, generate surrogate keys |
| **Load** | Insert each table into SQLite with `to_sql` (replace strategy) |
| **Validate** | Row-count checks, FK orphan checks, null-PK checks |

## 0. Imports & Configuration

In [10]:
import sqlite3
import os
import logging
from datetime import datetime

import pandas as pd

# ---------------------------------------------------------------------------
# Logging — prints timestamp + level + message to stdout
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  [%(levelname)s]  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger('etl')

# ---------------------------------------------------------------------------
# Paths — adjust if your folder structure differs
# ---------------------------------------------------------------------------
BASE_DIR  = os.path.dirname(os.path.abspath('__file__'))
CSV_PATH  = os.path.join(BASE_DIR, '..','data', 'DataForAssessment.csv')
DB_PATH   = os.path.join(BASE_DIR, 'drilling_operations.db')

log.info('Configuration loaded.')
log.info(f'  CSV  : {CSV_PATH}')
log.info(f'  DB   : {DB_PATH}')

13:05:23  [INFO]  Configuration loaded.
13:05:23  [INFO]    CSV  : c:\Users\AEM-Mior\OneDrive - Aem Energy Solutions\Working File\5. PROJECT\2026\13-PYTHON FUNDAMENTAL\Python-Submission\Data-Journey-Kickstart-Python-Assessment\Mior_DataEngineering_Assessment\..\data\DataForAssessment.csv
13:05:23  [INFO]    DB   : c:\Users\AEM-Mior\OneDrive - Aem Energy Solutions\Working File\5. PROJECT\2026\13-PYTHON FUNDAMENTAL\Python-Submission\Data-Journey-Kickstart-Python-Assessment\Mior_DataEngineering_Assessment\drilling_operations.db


---
## Stage 1 — EXTRACT
Read the raw CSV and perform a basic data-quality inspection before any transformation.

In [11]:
log.info('=== EXTRACT ===')

# Read raw flat table — keep all columns as strings initially to avoid
# pandas silently misreading mixed-format dates/numbers.
df_raw = pd.read_csv(CSV_PATH, dtype=str)

log.info(f'Loaded {len(df_raw):,} rows x {len(df_raw.columns)} columns from CSV.')

# --- Shape & dtypes ---
print('\n--- Shape ---')
print(f'Rows: {df_raw.shape[0]}  |  Columns: {df_raw.shape[1]}')

print('\n--- Columns ---')
print(df_raw.columns.tolist())

# --- Missing value summary (columns with at least one NaN) ---
print('\n--- Missing Values (columns with nulls) ---')
missing = df_raw.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(missing.to_string() if not missing.empty else 'None')

# --- Duplicate rows ---
n_dupes = df_raw.duplicated().sum()
print(f'\n--- Duplicate Rows ---')
print(f'{n_dupes:,} exact duplicate rows detected.')

log.info('Extract complete.')

13:05:23  [INFO]  === EXTRACT ===
13:05:23  [INFO]  Loaded 138 rows x 27 columns from CSV.
13:05:23  [INFO]  Extract complete.



--- Shape ---
Rows: 138  |  Columns: 27

--- Columns ---
['PacName', 'RegionName', 'FieldName', 'WellName', 'WellType', 'RigName', 'RigType', 'WaterDepth', 'Year', 'ReportType', 'DocumentName', 'DocumentDate', 'SubmittedAt', 'SubmittedBy', 'AfeCost', 'AfeDays', 'SpudDate', 'WellStartDateTime', 'WellEndDateTime', 'FinalCost', 'FinalDays', 'WellNptPercentageWow', 'WellNptPercentage', 'CompletionCostPlan', 'CompletionCostActual', 'DrillingPlanWcpf', 'DrillingActualWcpf']

--- Missing Values (columns with nulls) ---
DrillingActualWcpf      138
DrillingPlanWcpf        138
CompletionCostActual    138
CompletionCostPlan      138
DocumentName            134
DocumentDate             55
RegionName                2
RigType                   2

--- Duplicate Rows ---
6 exact duplicate rows detected.


---
## Stage 2 — TRANSFORM
Each sub-cell transforms one table. Steps applied per table:
1. Select relevant columns from `df_raw`
2. Cast to correct dtypes
3. Drop duplicates
4. Sort for readability
5. Reset index
6. Generate surrogate key where required (yellow/orange in ERD)

In [12]:
# ---------------------------------------------------------------------------
# Helper: normalise a datetime column with mixed timezones → UTC-naive
# ---------------------------------------------------------------------------
def to_utc_naive(series: pd.Series) -> pd.Series:
    """Parse a string series with mixed tz offsets → timezone-naive UTC datetime."""
    return pd.to_datetime(series, utc=True).dt.tz_localize(None)


# ---------------------------------------------------------------------------
# Helper: log transform result
# ---------------------------------------------------------------------------
def log_transform(name: str, df: pd.DataFrame):
    log.info(f'  {name:<28} → {len(df):>4} rows  |  cols: {df.columns.tolist()}')


log.info('=== TRANSFORM ===')

13:05:23  [INFO]  === TRANSFORM ===


In [13]:
# ------------------------------------------------------------------
# PAC — natural PK: PacName
# ------------------------------------------------------------------
pac_table = (
    df_raw[['PacName']]
    .dropna(subset=['PacName'])       # PK cannot be null
    .drop_duplicates()
    .sort_values('PacName')
    .reset_index(drop=True)
)

log_transform('PAC', pac_table)
print(pac_table)

13:05:24  [INFO]    PAC                          →   14 rows  |  cols: ['PacName']


           PacName
0              COP
1           EMEPMI
2             HESS
3        JX NIPPON
4           MURPHY
5             NOEX
6             PCSB
7            PTTEP
8           REPSOL
9          ROC OIL
10    SEA HIBISCUS
11           SHELL
12  TEST PAC2 EDIT
13         VESTIGO


In [14]:
# ------------------------------------------------------------------
# REGION — natural PK: RegionName  |  FK: PacName
# ------------------------------------------------------------------
region_table = (
    df_raw[['RegionName', 'PacName']]
    .dropna(subset=['RegionName'])    # PK cannot be null
    .drop_duplicates()
    .sort_values('RegionName')
    .reset_index(drop=True)
)

log_transform('Region', region_table)
print(region_table)

13:05:24  [INFO]    Region                       →   18 rows  |  cols: ['RegionName', 'PacName']


   RegionName       PacName
0          PM          PCSB
1          PM       VESTIGO
2          PM        EMEPMI
3          PM          HESS
4          PM         SHELL
5          PM         PTTEP
6          SB          HESS
7          SB        REPSOL
8          SB  SEA HIBISCUS
9          SK     JX NIPPON
10         SK           COP
11         SK          PCSB
12         SK         SHELL
13         SK       ROC OIL
14         SK       VESTIGO
15         SK          NOEX
16         SK        MURPHY
17         SK         PTTEP


In [15]:
# ------------------------------------------------------------------
# FIELD — natural PK: FieldName  |  FK: RegionName
# ------------------------------------------------------------------
field_table = (
    df_raw[['FieldName', 'RegionName']]
    .dropna(subset=['FieldName'])     # PK cannot be null
    .drop_duplicates()
    .sort_values('FieldName')
    .reset_index(drop=True)
)

log_transform('Field', field_table)
print(field_table)

13:05:24  [INFO]    Field                        →   32 rows  |  cols: ['FieldName', 'RegionName']


             FieldName RegionName
0                ANGSI         PM
1         ANJUNG KECIL         SK
2                BELUM         SK
3              BENTARA         SK
4     BERGADING-C1 DEV         PM
5                BERYL         SK
6         BESAR-A4 DEV         PM
7            BULOH DEV        NaN
8    BUNGA KESUMBA DEV         SB
9            GAGAU EXP         SK
10   GAMUSUT KAKAP DEV         PM
11         GOREK A DEV         SK
12                  J4         SK
13         KAMOMIL EXP         SK
14          KCP-1 EXPL         SK
15          KECAPI EXP         SK
16            KERDAS-1         SK
17      KINABALU D DEV         SB
18      LANG LEBAH EXP         SK
19       LARUT-A26 DEV         PM
20              LAYANG         SK
21         M1-A111 DEV         SK
22    MACHINCANG-1 EXP         SK
23            PRPA DEV         SK
24         SALAM-3 EXP         SK
25  SEPAT DEPS DEV APP         PM
26            SERENDAH         SK
27          SOUTH ACIS         SK
28     ST JOSE

In [16]:
# ------------------------------------------------------------------
# RIG — natural PK: RigName
# ------------------------------------------------------------------
rig_table = (
    df_raw[['RigName', 'RigType']]
    .dropna(subset=['RigName'])       # PK cannot be null
    .drop_duplicates()
    .sort_values('RigName')
    .reset_index(drop=True)
)

log_transform('Rig', rig_table)
print(rig_table)

13:05:24  [INFO]    Rig                          →   29 rows  |  cols: ['RigName', 'RigType']


                RigName                                        RigType
0             ABAN VIII                                        JACK-UP
1   AQUA MARINE DRILLER                                        JACK UP
2          BORR GUNNLOD                                        JACK UP
3             BORR SAGA                                        JACK UP
4    DEEPWATER NAUTILUS                               Semi-Submersible
5              ENSCO-52                                        JACK UP
6         GALVESTON KEY  PLATFORM TENDER-ASSISTED SEMI-SUB TYPE VESSEL
7               GUNNLOD                                        JACK UP
8             HAKURYU-5                               Semi-Submersible
9         MAERSK VIKING                                     DRILL SHIP
10                 MIST                                        JACK UP
11             NAGA - 5                                        JACK-UP
12               NAGA 4                                            NaN
13    

In [17]:
# ------------------------------------------------------------------
# WELL — natural PK: WellName  |  FK: FieldName, RigName
# Type casts:
#   WaterDepth → float
#   Year       → Int64 (nullable integer)
#   SpudDate   → date (UTC-normalised, time component dropped)
#   WellStart/EndDateTime → datetime (UTC-normalised, tz-naive)
# ------------------------------------------------------------------
well_table = (
    df_raw[['WellName', 'FieldName', 'RigName', 'WellType',
            'WaterDepth', 'Year', 'SpudDate',
            'WellStartDateTime', 'WellEndDateTime']]
    .dropna(subset=['WellName'])      # PK cannot be null
    .drop_duplicates()
    .sort_values('WellName')
    .reset_index(drop=True)
    .assign(
        WaterDepth        = lambda x: pd.to_numeric(x['WaterDepth'], errors='coerce'),
        Year              = lambda x: pd.to_numeric(x['Year'], errors='coerce').astype('Int64'),
        SpudDate          = lambda x: to_utc_naive(x['SpudDate']).dt.date,
        WellStartDateTime = lambda x: to_utc_naive(x['WellStartDateTime']),
        WellEndDateTime   = lambda x: to_utc_naive(x['WellEndDateTime']),
    )
)

log_transform('Well', well_table)
print(well_table.to_string())

13:05:24  [INFO]    Well                         →   64 rows  |  cols: ['WellName', 'FieldName', 'RigName', 'WellType', 'WaterDepth', 'Year', 'SpudDate', 'WellStartDateTime', 'WellEndDateTime']


              WellName           FieldName              RigName                       WellType   WaterDepth  Year    SpudDate   WellStartDateTime     WellEndDateTime
0                AAZ 1           BULOH DEV      SAPURA ALLIANCE  APPRAISAL CUM DEV/DEVELOPMENT   500.000000  2025  2025-01-13 2025-01-13 16:58:00 2025-01-31 16:58:00
1               AJK-A1        ANJUNG KECIL  AQUA MARINE DRILLER  APPRAISAL CUM DEV/DEVELOPMENT    70.500000  2013  2013-09-14 2013-09-14 00:00:00 2013-11-11 04:45:00
2            AJK-A1ST1        ANJUNG KECIL                 MIST  APPRAISAL CUM DEV/DEVELOPMENT    70.500000  2019  2019-06-29 2019-06-19 00:00:00 2019-08-23 22:30:00
3               AJK-A2        ANJUNG KECIL                 MIST  APPRAISAL CUM DEV/DEVELOPMENT    70.500000  2013  2013-09-19 2013-09-18 15:30:00 2013-11-15 18:00:00
4            AJK-A2ST1        ANJUNG KECIL                 MIST  APPRAISAL CUM DEV/DEVELOPMENT    70.500000  2019  2019-07-12 2019-05-30 18:00:00 2019-08-31 00:30:00
5   

In [18]:
# ------------------------------------------------------------------
# AFE — surrogate PK: IDAFE (auto-assigned from index + 1)
# ------------------------------------------------------------------
afe_table = (
    df_raw[['WellName', 'AfeCost', 'AfeDays', 'FinalCost', 'FinalDays']]
    .dropna(subset=['AfeCost', 'AfeDays', 'FinalCost', 'FinalDays'], how='all')
    .drop_duplicates()
    .sort_values(['WellName', 'AfeCost', 'AfeDays', 'FinalCost', 'FinalDays'])
    .reset_index(drop=True)
    .assign(
        IDAFE     = lambda x: x.index + 1,
        AfeCost   = lambda x: pd.to_numeric(x['AfeCost'],   errors='coerce'),
        AfeDays   = lambda x: pd.to_numeric(x['AfeDays'],   errors='coerce'),
        FinalCost = lambda x: pd.to_numeric(x['FinalCost'], errors='coerce'),
        FinalDays = lambda x: pd.to_numeric(x['FinalDays'], errors='coerce'),
    )
    .pipe(lambda x: x[['IDAFE', 'WellName', 'AfeCost', 'AfeDays', 'FinalCost', 'FinalDays']])
)

log_transform('AFE', afe_table)
print(afe_table)

13:05:24  [INFO]    AFE                          →   64 rows  |  cols: ['IDAFE', 'WellName', 'AfeCost', 'AfeDays', 'FinalCost', 'FinalDays']


    IDAFE            WellName      AfeCost  AfeDays     FinalCost  FinalDays
0       1               AAZ 1   5000000.00    50.00  1.000000e+00   1.000000
1       2              AJK-A1  27818489.00    51.00  1.767084e+07  30.100000
2       3           AJK-A1ST1   8373993.00    22.20  8.990865e+06  29.450000
3       4              AJK-A2  26753912.00    48.00  1.664989e+07  32.000000
4       5           AJK-A2ST1   8100000.00    20.50  6.081416e+06  21.140000
..    ...                 ...          ...      ...           ...        ...
59     60          TEST-SHELL         0.00     0.00  5.000000e+06  65.000000
60     61  WEST LUTONG-A08ST3  13892863.88    62.84  1.455986e+07  70.468750
61     62  WEST LUTONG-A13ST2   9926255.83    34.07  8.624697e+06  37.770832
62     63             WPPB-14   5866029.56     8.48  9.160501e+06  24.291666
63     64       WPPB-15/15ST1   7999172.03    17.44  1.602606e+07  37.208332

[64 rows x 6 columns]


In [19]:
# ------------------------------------------------------------------
# WELLNPT — surrogate PK: IDWellNpt
# ------------------------------------------------------------------
wellnpt_table = (
    df_raw[['WellName', 'WellNptPercentageWow', 'WellNptPercentage']]
    .dropna(subset=['WellNptPercentageWow', 'WellNptPercentage'], how='all')
    .drop_duplicates()
    .sort_values(['WellName', 'WellNptPercentageWow', 'WellNptPercentage'])
    .reset_index(drop=True)
    .assign(
        IDWellNpt            = lambda x: x.index + 1,
        WellNptPercentageWow = lambda x: pd.to_numeric(x['WellNptPercentageWow'], errors='coerce'),
        WellNptPercentage    = lambda x: pd.to_numeric(x['WellNptPercentage'],    errors='coerce'),
    )
    .pipe(lambda x: x[['IDWellNpt', 'WellName', 'WellNptPercentageWow', 'WellNptPercentage']])
)

log_transform('WellNpt', wellnpt_table)
print(wellnpt_table)

13:05:24  [INFO]    WellNpt                      →   64 rows  |  cols: ['IDWellNpt', 'WellName', 'WellNptPercentageWow', 'WellNptPercentage']


    IDWellNpt            WellName  WellNptPercentageWow  WellNptPercentage
0           1               AAZ 1              1.000000           1.000000
1           2              AJK-A1              9.660000           9.660000
2           3           AJK-A1ST1             24.080000          24.080000
3           4              AJK-A2             18.000000          18.000000
4           5           AJK-A2ST1              6.600000           6.600000
..        ...                 ...                   ...                ...
59         60          TEST-SHELL             67.000000          50.000000
60         61  WEST LUTONG-A08ST3              8.001177           7.538803
61         62  WEST LUTONG-A13ST2             13.696009          12.934363
62         63             WPPB-14             34.905660          34.905660
63         64       WPPB-15/15ST1              1.259798           1.259798

[64 rows x 4 columns]


In [20]:
# ------------------------------------------------------------------
# WELLCOMPLETIONCOST — surrogate PK: IDWellCompletionCost
# ------------------------------------------------------------------
wellcompletioncost_table = (
    df_raw[['WellName', 'CompletionCostPlan', 'CompletionCostActual']]
    .dropna(subset=['CompletionCostPlan', 'CompletionCostActual'], how='all')
    .drop_duplicates()
    .sort_values(['WellName', 'CompletionCostPlan', 'CompletionCostActual'])
    .reset_index(drop=True)
    .assign(
        IDWellCompletionCost = lambda x: x.index + 1,
        CompletionCostPlan   = lambda x: pd.to_numeric(x['CompletionCostPlan'],   errors='coerce'),
        CompletionCostActual = lambda x: pd.to_numeric(x['CompletionCostActual'], errors='coerce'),
    )
    .pipe(lambda x: x[['IDWellCompletionCost', 'WellName', 'CompletionCostPlan', 'CompletionCostActual']])
)

log_transform('WellCompletionCost', wellcompletioncost_table)
print(wellcompletioncost_table)

13:05:24  [INFO]    WellCompletionCost           →    0 rows  |  cols: ['IDWellCompletionCost', 'WellName', 'CompletionCostPlan', 'CompletionCostActual']


Empty DataFrame
Columns: [IDWellCompletionCost, WellName, CompletionCostPlan, CompletionCostActual]
Index: []


In [21]:
# ------------------------------------------------------------------
# WELLDRILLING — surrogate PK: IDWellDrilling
# ------------------------------------------------------------------
welldrilling_table = (
    df_raw[['WellName', 'DrillingPlanWcpf', 'DrillingActualWcpf']]
    .dropna(subset=['DrillingPlanWcpf', 'DrillingActualWcpf'], how='all')
    .drop_duplicates()
    .sort_values(['WellName', 'DrillingPlanWcpf', 'DrillingActualWcpf'])
    .reset_index(drop=True)
    .assign(
        IDWellDrilling     = lambda x: x.index + 1,
        DrillingPlanWcpf   = lambda x: pd.to_numeric(x['DrillingPlanWcpf'],   errors='coerce'),
        DrillingActualWcpf = lambda x: pd.to_numeric(x['DrillingActualWcpf'], errors='coerce'),
    )
    .pipe(lambda x: x[['IDWellDrilling', 'WellName', 'DrillingPlanWcpf', 'DrillingActualWcpf']])
)

log_transform('WellDrilling', welldrilling_table)
print(welldrilling_table)

13:05:24  [INFO]    WellDrilling                 →    0 rows  |  cols: ['IDWellDrilling', 'WellName', 'DrillingPlanWcpf', 'DrillingActualWcpf']


Empty DataFrame
Columns: [IDWellDrilling, WellName, DrillingPlanWcpf, DrillingActualWcpf]
Index: []


In [22]:
# ------------------------------------------------------------------
# REPORT — surrogate PK: IDReport
# ------------------------------------------------------------------
report_table = (
    df_raw[['WellName', 'ReportType', 'DocumentName',
            'DocumentDate', 'SubmittedAt', 'SubmittedBy']]
    .dropna(subset=['ReportType', 'DocumentName', 'DocumentDate',
                    'SubmittedAt', 'SubmittedBy'], how='all')
    .drop_duplicates()
    .sort_values(['WellName', 'ReportType', 'DocumentName',
                  'DocumentDate', 'SubmittedAt', 'SubmittedBy'])
    .reset_index(drop=True)
    .assign(
        IDReport     = lambda x: x.index + 1,
        DocumentDate = lambda x: to_utc_naive(x['DocumentDate']),
        SubmittedAt  = lambda x: to_utc_naive(x['SubmittedAt']),
    )
    .pipe(lambda x: x[['IDReport', 'WellName', 'ReportType', 'DocumentName',
                        'DocumentDate', 'SubmittedAt', 'SubmittedBy']])
)

log_transform('Report', report_table)
print(report_table)

13:05:24  [INFO]    Report                       →  132 rows  |  cols: ['IDReport', 'WellName', 'ReportType', 'DocumentName', 'DocumentDate', 'SubmittedAt', 'SubmittedBy']


     IDReport            WellName ReportType  \
0           1               AAZ 1        FWR   
1           2               AAZ 1       NOOP   
2           3              AJK-A1        FWR   
3           4              AJK-A1       NOOP   
4           5           AJK-A1ST1        FWR   
..        ...                 ...        ...   
127       128  WEST LUTONG-A13ST2       NOOP   
128       129             WPPB-14        FWR   
129       130             WPPB-14       NOOP   
130       131       WPPB-15/15ST1        FWR   
131       132       WPPB-15/15ST1       NOOP   

                                 DocumentName            DocumentDate  \
0    0348dacf-d28d-4988-b664-b1e8e440b0a0.pdf 2025-01-13 16:57:29.562   
1    b3c2ae91-cdc3-4532-b1f3-efa7eaf9fd09.pdf 2025-01-09 00:00:00.000   
2                                         NaN 2013-12-20 00:00:00.000   
3                                         NaN 2013-09-06 00:00:00.000   
4                                         NaN 2019-10-22 0

---
## Stage 3 — LOAD
Insert all transformed DataFrames into `drilling_operations.db` using `pandas.to_sql`.

- `if_exists='replace'` drops & recreates the table on each run (idempotent)
- `index=False` prevents pandas writing its own integer index as a column
- Tables are loaded **in FK dependency order** (parent before child)

> **Load order:** PAC → Region → Field → Rig → Well → AFE → WellNpt → WellCompletionCost → WellDrilling → Report

In [23]:
log.info('=== LOAD ===')

# Tables in FK dependency order (parents first)
TABLES = [
    ('PAC',                  pac_table),
    ('Region',               region_table),
    ('Field',                field_table),
    ('Rig',                  rig_table),
    ('Well',                 well_table),
    ('AFE',                  afe_table),
    ('WellNpt',              wellnpt_table),
    ('WellCompletionCost',   wellcompletioncost_table),
    ('WellDrilling',         welldrilling_table),
    ('Report',               report_table),
]

load_summary = []

try:
    conn = sqlite3.connect(DB_PATH)
    # Disable FK enforcement during bulk load to avoid ordering issues,
    # then re-enable for subsequent queries.
    conn.execute('PRAGMA foreign_keys = OFF;')

    for table_name, df in TABLES:
        try:
            df.to_sql(
                name      = table_name,
                con       = conn,
                if_exists = 'replace',   # idempotent: drop & recreate on each run
                index     = False,       # do not write pandas index as a column
            )
            log.info(f'  Loaded {len(df):>4} rows → {table_name}')
            load_summary.append({'Table': table_name, 'Rows Loaded': len(df), 'Status': 'OK'})

        except Exception as e:
            log.error(f'  FAILED to load {table_name}: {e}')
            load_summary.append({'Table': table_name, 'Rows Loaded': 0, 'Status': f'ERROR: {e}'})

    conn.execute('PRAGMA foreign_keys = ON;')
    conn.commit()
    log.info('All tables committed to database.')

except Exception as e:
    log.error(f'Database connection failed: {e}')
    raise

finally:
    conn.close()

# --- Load summary ---
print('\n--- Load Summary ---')
print(pd.DataFrame(load_summary).to_string(index=False))

13:05:24  [INFO]  === LOAD ===
13:05:24  [INFO]    Loaded   14 rows → PAC
13:05:24  [INFO]    Loaded   18 rows → Region
13:05:24  [INFO]    Loaded   32 rows → Field
13:05:24  [INFO]    Loaded   29 rows → Rig
13:05:24  [INFO]    Loaded   64 rows → Well
13:05:24  [INFO]    Loaded   64 rows → AFE
13:05:24  [INFO]    Loaded   64 rows → WellNpt
13:05:24  [INFO]    Loaded    0 rows → WellCompletionCost
13:05:24  [INFO]    Loaded    0 rows → WellDrilling
13:05:24  [INFO]    Loaded  132 rows → Report
13:05:24  [INFO]  All tables committed to database.



--- Load Summary ---
             Table  Rows Loaded Status
               PAC           14     OK
            Region           18     OK
             Field           32     OK
               Rig           29     OK
              Well           64     OK
               AFE           64     OK
           WellNpt           64     OK
WellCompletionCost            0     OK
      WellDrilling            0     OK
            Report          132     OK


---
## Stage 4 — VALIDATE
Run post-load checks directly against the SQLite database:

1. **Row count check** — DB row count matches DataFrame row count
2. **Null PK check** — no NULL values in any primary key column
3. **FK orphan check** — no child rows referencing a non-existent parent

In [24]:
log.info('=== VALIDATE ===')

conn = sqlite3.connect(DB_PATH)

# ------------------------------------------------------------------
# Check 1: Row counts — DB vs DataFrame
# ------------------------------------------------------------------
print('--- Check 1: Row Counts ---')
print(f'  {"Table":<28} {"DataFrame":>12} {"DB":>8} {"Match":>8}')
print(f'  {"-"*28} {"-"*12} {"-"*8} {"-"*8}')

all_match = True
for table_name, df in TABLES:
    db_count = conn.execute(f'SELECT COUNT(*) FROM "{table_name}"').fetchone()[0]
    df_count = len(df)
    match    = '✓' if db_count == df_count else '✗ MISMATCH'
    if db_count != df_count:
        all_match = False
    print(f'  {table_name:<28} {df_count:>12,} {db_count:>8,} {match:>8}')

log.info('Row count check: ' + ('ALL PASSED ✓' if all_match else 'FAILURES DETECTED ✗'))

13:05:25  [INFO]  === VALIDATE ===
13:05:25  [INFO]  Row count check: ALL PASSED ✓


--- Check 1: Row Counts ---
  Table                           DataFrame       DB    Match
  ---------------------------- ------------ -------- --------
  PAC                                    14       14        ✓
  Region                                 18       18        ✓
  Field                                  32       32        ✓
  Rig                                    29       29        ✓
  Well                                   64       64        ✓
  AFE                                    64       64        ✓
  WellNpt                                64       64        ✓
  WellCompletionCost                      0        0        ✓
  WellDrilling                            0        0        ✓
  Report                                132      132        ✓


In [25]:
# ------------------------------------------------------------------
# Check 2: Null Primary Key check
# ------------------------------------------------------------------
PK_MAP = {
    'PAC':                'PacName',
    'Region':             'RegionName',
    'Field':              'FieldName',
    'Rig':                'RigName',
    'Well':               'WellName',
    'AFE':                'IDAFE',
    'WellNpt':            'IDWellNpt',
    'WellCompletionCost': 'IDWellCompletionCost',
    'WellDrilling':       'IDWellDrilling',
    'Report':             'IDReport',
}

print('\n--- Check 2: Null Primary Keys ---')
pk_pass = True
for table, pk_col in PK_MAP.items():
    null_count = conn.execute(
        f'SELECT COUNT(*) FROM "{table}" WHERE "{pk_col}" IS NULL'
    ).fetchone()[0]
    status = '✓' if null_count == 0 else f'✗ {null_count} NULL PKs'
    if null_count > 0:
        pk_pass = False
    print(f'  {table:<28} PK={pk_col:<25} {status}')

log.info('Null PK check: ' + ('ALL PASSED ✓' if pk_pass else 'FAILURES DETECTED ✗'))

13:05:25  [INFO]  Null PK check: ALL PASSED ✓



--- Check 2: Null Primary Keys ---
  PAC                          PK=PacName                   ✓
  Region                       PK=RegionName                ✓
  Field                        PK=FieldName                 ✓
  Rig                          PK=RigName                   ✓
  Well                         PK=WellName                  ✓
  AFE                          PK=IDAFE                     ✓
  WellNpt                      PK=IDWellNpt                 ✓
  WellCompletionCost           PK=IDWellCompletionCost      ✓
  WellDrilling                 PK=IDWellDrilling            ✓
  Report                       PK=IDReport                  ✓


In [26]:
# ------------------------------------------------------------------
# Check 3: Foreign Key Orphan check
# ------------------------------------------------------------------
FK_CHECKS = [
    # (child_table,  child_fk_col,   parent_table,  parent_pk_col)
    ('Region',             'PacName',    'PAC',    'PacName'),
    ('Field',              'RegionName', 'Region', 'RegionName'),
    ('Well',               'FieldName',  'Field',  'FieldName'),
    ('Well',               'RigName',    'Rig',    'RigName'),
    ('AFE',                'WellName',   'Well',   'WellName'),
    ('WellNpt',            'WellName',   'Well',   'WellName'),
    ('WellCompletionCost', 'WellName',   'Well',   'WellName'),
    ('WellDrilling',       'WellName',   'Well',   'WellName'),
    ('Report',             'WellName',   'Well',   'WellName'),
]

print('\n--- Check 3: FK Orphan Rows ---')
fk_pass = True
for child_tbl, child_col, parent_tbl, parent_col in FK_CHECKS:
    orphan_count = conn.execute(f"""
        SELECT COUNT(*)
        FROM   "{child_tbl}" c
        WHERE  c."{child_col}" IS NOT NULL
          AND  c."{child_col}" NOT IN (
                   SELECT "{parent_col}" FROM "{parent_tbl}"
               )
    """).fetchone()[0]
    status = '✓' if orphan_count == 0 else f'✗ {orphan_count} orphans'
    if orphan_count > 0:
        fk_pass = False
    print(f'  {child_tbl}.{child_col:<20} → {parent_tbl}.{parent_col:<15} {status}')

log.info('FK orphan check: ' + ('ALL PASSED ✓' if fk_pass else 'FAILURES DETECTED ✗'))

conn.close()

13:05:25  [INFO]  FK orphan check: ALL PASSED ✓



--- Check 3: FK Orphan Rows ---
  Region.PacName              → PAC.PacName         ✓
  Field.RegionName           → Region.RegionName      ✓
  Well.FieldName            → Field.FieldName       ✓
  Well.RigName              → Rig.RigName         ✓
  AFE.WellName             → Well.WellName        ✓
  WellNpt.WellName             → Well.WellName        ✓
  WellCompletionCost.WellName             → Well.WellName        ✓
  WellDrilling.WellName             → Well.WellName        ✓
  Report.WellName             → Well.WellName        ✓


In [27]:
# ------------------------------------------------------------------
# Final summary
# ------------------------------------------------------------------
print('\n' + '='*55)
print('  ETL PIPELINE COMPLETE')
print('='*55)
print(f'  Timestamp : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'  Database  : {DB_PATH}')
print(f'  Row count : {"PASS ✓" if all_match else "FAIL ✗"}')
print(f'  Null PKs  : {"PASS ✓" if pk_pass  else "FAIL ✗"}')
print(f'  FK orphans: {"PASS ✓" if fk_pass  else "FAIL ✗"}')
overall = all([all_match, pk_pass, fk_pass])
print(f'  Overall   : {"ALL CHECKS PASSED ✓" if overall else "ISSUES DETECTED — review above ✗"}')
print('='*55)


  ETL PIPELINE COMPLETE
  Timestamp : 2026-05-28 13:05:25
  Database  : c:\Users\AEM-Mior\OneDrive - Aem Energy Solutions\Working File\5. PROJECT\2026\13-PYTHON FUNDAMENTAL\Python-Submission\Data-Journey-Kickstart-Python-Assessment\Mior_DataEngineering_Assessment\drilling_operations.db
  Row count : PASS ✓
  Null PKs  : PASS ✓
  FK orphans: PASS ✓
  Overall   : ALL CHECKS PASSED ✓
